# AML Observability Walkthrough

Dieses Notebook zeigt den Beyond-AI-Observability-PoC mit eingecheckten Fixture-Daten, Erklaerungen fuer Alert- und Non-Alert-Faelle sowie einer einfachen Retention-Entscheidung.

In [ ]:
import json
from pathlib import Path

from observability.collector import InMemoryTraceCollector
from observability.models import CaseDisposition, TransactionRecord
from observability.pipeline import run_transaction
from observability.privacy import apply_trace_retention
from observability.queries import explain_what_changed, explain_why_flagged, explain_why_not_flagged

DATA_DIR = Path('../data').resolve()
transactions_path = DATA_DIR / 'synthetic_transactions.jsonl'
cases_path = DATA_DIR / 'synthetic_cases.jsonl'


In [ ]:
def read_jsonl(path: Path):
    with path.open('r', encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]


transactions = [TransactionRecord.from_dict(row) for row in read_jsonl(transactions_path)]
cases = {
    row['transaction_id']: CaseDisposition.from_dict({k: v for k, v in row.items() if k != 'transaction_id'})
    for row in read_jsonl(cases_path)
}

len(transactions), sorted(cases)


In [ ]:
collector = InMemoryTraceCollector()
traces = []

for transaction in transactions:
    trace = run_transaction(
        transaction,
        collector,
        disposition=cases.get(transaction.transaction_id),
        auto_case_feedback=transaction.transaction_id not in cases,
    )
    traces.append(trace)

[(trace.trace_id, trace.has_alert(), len(trace.events)) for trace in traces]


In [ ]:
flagged_trace = next(trace for trace in traces if trace.has_alert())
non_alert_trace = next(trace for trace in traces if not trace.has_alert())

flagged_explanation = explain_why_flagged(flagged_trace)
non_alert_explanation = explain_why_not_flagged(non_alert_trace)

print(flagged_explanation.answer)
print(non_alert_explanation.answer)


In [ ]:
retention_decision, retained_trace = apply_trace_retention(non_alert_trace)
delta_explanation = explain_what_changed(traces[0], traces[1])

print(retention_decision)
print('retained event count:', len(retained_trace.events))
print(delta_explanation.answer)
